<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/projects/02-llm-basics/01-prompt-engineering-basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guidelines for Prompting
In this notebook, we'll practice two prompting principles and their related tactics in order to write effective prompts for large language models.

In [6]:
!pip install groq
from groq import Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.3 MB/s eta 0:00:00


### Set up the Groq API Key
To use the Groq API, you'll need an API key. If you don't already have one, create a key on the [Groq website](https://groq.com/).
In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `groq_key`. Then pass the key to the client.

In [10]:
# =============================================
# CELL 1: Groq Setup with Streaming
# =============================================

!pip install -q groq

from groq import Groq
from google.colab import userdata

# Load API key from Colab secrets
GROQ_API_KEY = userdata.get('groq_key')
client = Groq(api_key=GROQ_API_KEY)

# # Make the API call
# completion = client.chat.completions.create(
#     model="llama-3.3-70b-versatile",
#     messages=[
#         {
#             "role": "user",
#             "content": "What is prompt engineering? Explain it in 3 sentences."
#         }
#     ],
#     temperature=1,
#     max_completion_tokens=1024,
#     top_p=1,
#     stream=True,
#     stop=None
# )

# # Print the streaming response
# print("Response:\n")
# for chunk in completion:
#     print(chunk.choices[0].delta.content or "", end="")

Response:

Prompt engineering is the process of designing and optimizing text prompts to elicit specific, desired responses from artificial intelligence (AI) models, such as language generators or chatbots. This involves crafting input text that is clear, concise, and well-structured to guide the AI model towards producing accurate, relevant, and useful output. By carefully refining the prompts, developers and users can improve the performance and effectiveness of AI systems, enabling them to better understand and respond to human input.

In [14]:
def get_completion(prompt, model="llama-3.3-70b-versatile"):
  messages = [{"role": "user", "content": prompt}]
  response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=1,)
  return response.choices[0].message.content

## Prompting Principles
- **Principle 1: Write clear and specific instructions**
- **Principle 2: Give the model time to “think”**

### Tactics

#### Tactic 1: Use delimiters to clearly indicate distinct parts of the input
- Delimiters can be anything like: ```, """, < >, `<tag> </tag>`, `:`

In [27]:
text = (
    f"For effective interaction with large language models, clarity and precision "
    f"in your instructions are paramount. This approach helps guide the model "
    f"toward the desired output and significantly reduces the likelihood of "
    f"irrelevant or erroneous responses. It's important to understand that "
    f"a clear prompt doesn't necessarily mean a short one; often, more extensive "
    f"prompts offer better context and detail, leading to richer and more "
    f"pertinent outputs from the model."
)
prompt = (
    f"Summarize the text delimited by triple backticks "
    f"into a single sentence.\n"
    f"```\n{text}\n```"
)
response = get_completion(prompt)
print(response)

For effective interaction with large language models, it is crucial to provide clear and precise instructions, which can be extensive, to guide the model towards the desired output and reduce the likelihood of irrelevant responses.


#### Tactic 2: Ask for a structured output
- JSON, HTML

In [18]:
prompt= f"""Generate a list of three made-up book titles
with their authors and genres. Provide them in JSON format
with the following keys: book_id,title,author,genre."""

response = get_completion(prompt)
print(response)

```json
[
  {
    "book_id": 1,
    "title": "The Shadows of Elyria",
    "author": "A.M. Winters",
    "genre": "Fantasy"
  },
  {
    "book_id": 2,
    "title": "Beyond the Last Horizon",
    "author": "Ethan J. Blackwood",
    "genre": "Science Fiction"
  },
  {
    "book_id": 3,
    "title": "Whispers in the Dark Night",
    "author": "Lila R. Grey",
    "genre": "Mystery"
  }
]
```


#### Tactic 3: Ask the model to check whether conditions are satisfied

In [28]:
text_1 = f"""
Preparing a simple sandwich is straightforward! First, gather your ingredients: two
slices of bread, your preferred filling (e.g., ham and cheese), and any condiments.
Next, spread a condiment on one slice of bread if desired. Place your filling on that
slice. Cover with the second slice of bread. If you want, cut it diagonally. Enjoy your
quick meal!
"""
prompt = f"""
Examine the text provided within triple backticks.
If it presents a series of procedural steps, reformulate them into this structured format:

Step 1 - ...
Step 2 - …
…
Step N - …

Should the text lack any instructional sequence,
then simply state "No procedural steps identified."

```{text_1}```
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Gather your ingredients, including two slices of bread, your preferred filling, and any condiments.
Step 2 - Spread a condiment on one slice of bread if desired.
Step 3 - Place your filling on the slice with the condiment.
Step 4 - Cover the filling with the second slice of bread.
Step 5 - Cut the sandwich diagonally if desired.
Step 6 - Enjoy your quick meal.


In [29]:
text_2 = f"""
The evening sky was painted with hues of orange and purple, a gentle breeze rustling
through the leaves of ancient oaks. A lone owl hooted softly in the distance, its call
echoing through the tranquil forest. Stars began to emerge, one by one, twinkling
against the deepening indigo. The air carried the scent of pine and damp earth,
a peaceful conclusion to the day.
"""
prompt = f"""
Examine the text provided within triple backticks.
If it presents a series of procedural steps, reformulate them into this structured format:

Step 1 - ...
Step 2 - …
…
Step N - …

Should the text lack any instructional sequence,
then simply state "No procedural steps identified."

```{text_2}```
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No procedural steps identified.


#### Tactic 4: "Few-shot" prompting

In [30]:
prompt = f"""
Your task is to answer in a consistent style.

<child>: Explain creativity to me.

<grandparent>: Creativity is the art of seeing the invisible, shaping the intangible, and
bringing forth the unheard. It is the dance of imagination with possibility, often
sparked by silence and nurtured by curiosity.

<child>: Explain curiosity to me.
"""
response = get_completion(prompt)
print(response)

<grandparent>: Curiosity is the gentle whisper of wonder, the soft rustle of leaves that stirs the mind, and the eager heartbeat that beckons discovery. It is the bridge that spans the chasm between the known and the unknown, inviting us to venture into the unexplored and to unravel the mysteries that surround us, like a masterful weaver threading together the tapestry of experience and knowledge.


### Principle 2: Give the model time to “think”

#### Tactic 1: Specify the steps required to complete a task

In [31]:
text = f"""
In a quaint forest, young siblings Lily and Tom ventured forth on an adventure to gather
wild berries from the Whispering Woods. As they skipped along, singing cheerful tunes,
a mishap occurred—Lily stumbled over an exposed root and tumbled down a small incline,
with Tom bravely rushing to her aid. Although a bit bruised, they returned home to
warm embraces. Despite the minor setback, their adventurous spirits remained high,
and they continued to explore with renewed delight.
"""
# example 1
prompt_1 = f"""
Execute these tasks:
1 - Summarize the provided text, delineated by triple
backticks, into one sentence.
2 - Translate this summary into Spanish.
3 - Identify and list each proper noun found in the Spanish summary.
4 - Generate a JSON object containing the keys: spanish_summary, num_names.

Present your responses separated by newlines.

Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
The young siblings ventured into the Whispering Woods to gather wild berries, but their adventure was briefly interrupted when one of them stumbled and fell, although they returned home safely and continued to explore with delight.

La traducción al español de la anterior oración es: En el bosque de Whispering Woods, los hermanos Lily y Tom se lanzaron a una aventura para recoger bayas silvestres, pero su aventura se vio interrumpida brevemente cuando uno de ellos tropezó y cayó, aunque regresaron a casa sanos y salvos y continuaron explorando con deleite.

Los nombres propios encontrados en la oración en español son: Whispering Woods, Lily, Tom.

```
{
  "spanish_summary": "En el bosque de Whispering Woods, los hermanos Lily y Tom se lanzaron a una aventura para recoger bayas silvestres, pero su aventura se vio interrumpida brevemente cuando uno de ellos tropezó y cayó, aunque regresaron a casa sanos y salvos y continuaron explorando con deleite.",
  "num_name

#### Ask for output in a specified format

In [32]:
prompt_2 = f"""
Your task is to perform the following actions:
1 - Summarize the following text delimited by
  <> with 1 sentence.
2 - Translate the summary into Spanish.
3 - List each proper noun in the Spanish summary.
4 - Output a json object that contains the
  following keys: spanish_summary, num_nouns.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Proper Nouns: <list of proper nouns in summary>
Output JSON: <json with summary and num_nouns>

Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)


Completion for prompt 2:
Summary: Young siblings Lily and Tom ventured into the Whispering Woods to gather berries, but their adventure was briefly interrupted when Lily stumbled and fell, although they returned home safely and continued to explore.
Translation: Los jóvenes hermanos Lily y Tom se adentraron en el bosque Susurros para recoger bayas, pero su aventura se vio brevemente interrumpida cuando Lily tropezó y cayó, aunque regresaron a casa sanos y salvos y continuaron explorando.
Proper Nouns: ['Lily', 'Tom', 'el bosque Susurros'] 
Note: 'el bosque Susurros' is the translation of 'Whispering Woods'.
Output JSON: {"spanish_summary": "Los jóvenes hermanos Lily y Tom se adentraron en el bosque Susurros para recoger bayas, pero su aventura se vio brevemente interrumpida cuando Lily tropezó y cayó, aunque regresaron a casa sanos y salvos y continuaron explorando.", "num_nouns": 3}


#### Tactic 2: Instruct the model to work out its own solution before rushing to a conclusion

In [33]:
prompt = f"""
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need
 help working out the financials.
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost
me a flat $100k per year, and an additional $10 / square
foot
What is the total cost for the first year of operations
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""
response = get_completion(prompt)
print(response)

To determine if the student's solution is correct, let's break down the costs:

1. Land cost: The land costs $100 per square foot, so the total land cost is indeed $100x, where x is the size of the installation in square feet.
2. Solar panel cost: The solar panels cost $250 per square foot, so the total solar panel cost is indeed $250x.
3. Maintenance cost: The maintenance contract has a flat cost of $100,000 per year, and an additional $10 per square foot. Therefore, the total maintenance cost is $100,000 + $10x, not $100,000 + $100x.

The correct total maintenance cost is $100,000 + $10x, not $100,000 + $100x. The student incorrectly added $100x instead of $10x for the maintenance cost per square foot.

The correct total cost is:
100x (land cost) + 250x (solar panel cost) + 100,000 (flat maintenance cost) + 10x (maintenance cost per square foot)
= 100x + 250x + 100,000 + 10x
= 360x + 100,000

Therefore, the student's solution is incorrect. The correct total cost for the first year of

In [34]:
prompt = f"""
Your task is to determine if the student's solution \
is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem including the final total.
- Then compare your solution to the student's solution \
and evaluate if the student's solution is correct or not.
Don't decide if the student's solution is correct until
you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution \
just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help \
working out the financials.
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
```
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)

<>:61: SyntaxWarning: invalid escape sequence '\ '
<>:61: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_4797/1709959923.py:61: SyntaxWarning: invalid escape sequence '\ '


Question:
```
I'm building a solar power installation and I need help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost me a flat $100k per year, and an additional $10 / square foot
What is the total cost for the first year of operations as a function of the number of square feet.
```
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
To calculate the total cost for the first year of operations, let's break down the costs:

1. Land cost: The cost of land is $100 per square foot, so for x square feet, the land cost is 100x.
2. Solar panel cost: The cost of solar panels is $250 per square foot, so for x square feet, the solar panel cost is 250x.
3. Maintenance cost: The maintenance c

## Model Limitations: Hallucinations
- Boie is a real company, the product name is not real.

In [36]:
prompt = f"""
Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
"""
response = get_completion(prompt)
print(response)

The AeroGlide UltraSlim Smart Toothbrush by Boie is a high-tech, slim, and lightweight oral care device designed to provide an efficient and effective brushing experience. Here are some key features:

1. **UltraSlim Design**: The toothbrush has a slim and ergonomic design, making it comfortable to hold and easy to maneuver in the mouth.
2. **Advanced Brushing Technology**: The AeroGlide features advanced brushing technology, including sonic vibrations and gentle brushing motions, to remove plaque and food particles effectively.
3. **Smart Sensors**: The toothbrush is equipped with smart sensors that track brushing habits, including duration, frequency, and pressure. This data can be synced to a mobile app, providing users with personalized feedback and recommendations for improvement.
4. **Quad-Speed Control**: The AeroGlide offers four adjustable speed settings, allowing users to customize their brushing experience based on their individual needs and preferences.
5. **Long-Lasting Bat